In [2]:
import numpy as np
from analysis import *
from ellipsoid_fit import *

import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from scipy.spatial.transform import Rotation
np.set_printoptions(precision=3, suppress=True)

In [3]:
%matplotlib widget

In [5]:
def load_data():
    data = process_file_to_raw_data("magacc_readings_8.txt")
    mag = data[:,:3]
    acc = data[:,3:]

    norm = np.linalg.norm(acc, 2, axis=1)
    idcs = np.isclose(norm, 9.8, atol=0.4)

    mag = mag[idcs]
    acc = acc[idcs]
    return mag, acc



In [19]:
mag, acc = load_data()

In [20]:
import numpy as np

class RotationFinder:
    def __init__(self):
        self.G = np.zeros((9, 9))
        self.h = np.zeros(9)
        self.R = np.eye(3)
    
    def add_sample(self, mag, acc):
        m = mag / np.linalg.norm(mag)
        a = acc / np.linalg.norm(acc)
        k = np.kron(m, a)
        self.G += np.outer(k, k)
        self.h += k
    
    def solve(self):
        r = np.linalg.solve(self.G, self.h)
        R_raw = r.reshape(3, 3, order='F')
        U, _, Vt = np.linalg.svd(R_raw)
        self.R = U @ Vt
        if np.linalg.det(self.R) < 0:
            U[:, -1] *= -1
            self.R = U @ Vt
        return self.R

In [21]:
RF = RotationFinder()
for m,a in zip(mag, acc):
    RF.add_sample(m,a)
R = RF.solve()
R

array([[ 0.774, -0.556,  0.304],
       [-0.631, -0.718,  0.294],
       [ 0.055, -0.419, -0.906]])

In [32]:
A = np.array([ np.sum((R @ m) * a / np.linalg.norm(m) / np.linalg.norm(a)) for m,a in zip(mag, acc)])
A

array([0.707, 0.695, 0.694, 0.518, 0.711, 0.691, 0.708, 0.801, 0.707,
       0.457, 0.69 , 0.709, 0.783, 0.897, 0.535, 0.823, 0.662, 0.785,
       0.89 , 0.873, 0.638, 0.552, 0.742, 0.983, 0.919, 0.665, 0.987,
       0.478, 0.63 , 0.691, 0.798, 0.645, 0.83 , 0.646, 0.807, 0.746,
       0.929, 0.567, 0.514, 0.935, 0.648, 0.678, 0.735, 0.92 , 0.486,
       0.476, 0.819, 0.717, 0.7  , 0.75 , 0.569, 0.694, 0.576, 0.79 ,
       0.639, 0.68 , 0.796, 0.851, 0.887, 0.526, 0.578, 0.879, 0.843,
       0.616, 0.664, 0.524, 0.915, 0.922, 0.893, 0.758, 0.423, 0.75 ,
       0.709, 0.787, 0.832, 0.787, 0.722, 0.598, 0.849, 0.687, 0.929,
       0.811, 0.855, 0.929, 0.796, 0.652, 0.802, 0.561, 0.915, 0.605,
       0.878, 0.693, 0.511, 0.76 , 0.778, 0.528, 0.756, 0.457, 0.809,
       0.76 , 0.797, 0.527, 0.561, 0.517, 0.796, 0.662, 0.867, 0.978,
       0.491, 0.654, 0.44 , 0.756, 0.525, 0.717, 0.504, 0.534, 0.709,
       0.541, 0.492, 0.489, 0.792, 0.709, 0.781, 0.687, 0.681, 0.644,
       0.707, 0.608,